In [0]:

import os
import re
import json
import hashlib
from datetime import datetime, timezone
from urllib.parse import urljoin, unquote

import requests
from bs4 import BeautifulSoup
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType,
)

# COMMAND ----------

# =============================================================================
# CONFIGURAÇÃO
# =============================================================================

CATALOG = "voebem"
SCHEMA = "bronze"
VOLUME = "arquivos"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CONTROL_TABLE = f"{CATALOG}.{SCHEMA}.anac_ingestao_controle"

ANOS_VRA = [2025, 2026]   # <- ajuste aqui quando virar 2027

TIMEOUT = 300
HEADERS = {
    "User-Agent": "voebem-pipeline/1.0 (ingestao dados abertos ANAC)",
}

BASE = "https://sistemas.anac.gov.br/dadosabertos/"

DATASETS = [
    {
        "nome": "vra",
        "listing_url": BASE + "Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/"
                              "Voo%20Regular%20Ativo%20%28VRA%29/",
        "pattern": r"^VRA_(\d{4})(\d{1,2})\.csv$",
        "max_depth": 2,        # nivel 1 = ano, nivel 2 = mes
        "filtro_anos": ANOS_VRA,
        "subpasta": "vra",
    },
    {
        "nome": "empresas_estrangeiras",
        "listing_url": BASE + "Operador%20A%C3%A9reo/"
                              "Empresas%20Aereas%20Estrangeiras/",
        "pattern": r"^pda_empresas_aereas_estrangeiros\.csv$",
        "max_depth": 0,
        "filtro_anos": None,
        "subpasta": "referencia",
    },
    {
        "nome": "empresas_nacionais",
        "listing_url": BASE + "Operador%20A%C3%A9reo/"
                              "Empresas%20Aereas%20Nacionais/",
        "pattern": r"^pda_empresas_aereas_nacionais\.csv$",
        "max_depth": 0,
        "filtro_anos": None,
        "subpasta": "referencia",
    },
    {
        "nome": "aerodromos_publicos",
        "listing_url": BASE + "Aerodromos/Aer%C3%B3dromos%20P%C3%BAblicos/"
                              "Lista%20de%20aer%C3%B3dromos%20p%C3%BAblicos/",
        "pattern": r"^AerodromosPublicos\.csv$",
        "max_depth": 0,
        "filtro_anos": None,
        "subpasta": "referencia",
    },
]

# COMMAND ----------

# =============================================================================
# TABELA DE CONTROLE
# =============================================================================

CONTROL_SCHEMA = StructType([
    StructField("dataset", StringType()),
    StructField("file_name", StringType()),
    StructField("file_url", StringType()),
    StructField("content_length", LongType()),
    StructField("last_modified", StringType()),
    StructField("etag", StringType()),
    StructField("sha256", StringType()),
    StructField("loaded_at", TimestampType()),
])

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
        dataset        STRING,
        file_name      STRING,
        file_url       STRING,
        content_length BIGINT,
        last_modified  STRING,
        etag           STRING,
        sha256         STRING,
        loaded_at      TIMESTAMP
    ) USING DELTA
""")


def carregar_controle() -> dict:
    """{file_name: (content_length, last_modified, etag)} do load mais recente."""
    w = Window.partitionBy("file_name").orderBy(F.col("loaded_at").desc())
    df = (
        spark.table(CONTROL_TABLE)
        .withColumn("rn", F.row_number().over(w))
        .filter("rn = 1")
    )
    return {
        r["file_name"]: (r["content_length"], r["last_modified"], r["etag"])
        for r in df.collect()
    }


def registrar_controle(registros: list):
    if registros:
        (spark.createDataFrame(registros, schema=CONTROL_SCHEMA)
              .write.mode("append").saveAsTable(CONTROL_TABLE))

# COMMAND ----------

# =============================================================================
# DESCOBERTA DE ARQUIVOS
# =============================================================================

def listar_entradas(url: str) -> list:
    """Retorna [(nome, url_absoluta, eh_pasta)] de um Index of do Apache."""
    resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    entradas = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith(("?", "#")) or href in ("../", "/"):
            continue
        url_abs = urljoin(url, href)
        nome = unquote(href.rstrip("/").split("/")[-1])
        entradas.append((nome, url_abs, href.endswith("/")))
    return entradas


def coletar_arquivos(ds: dict) -> list:
    """Percorre a árvore até max_depth e devolve [(file_name, file_url)]."""
    regex = re.compile(ds["pattern"], re.IGNORECASE)
    encontrados = []

    def caminhar(url: str, depth: int):
        try:
            entradas = listar_entradas(url)
        except Exception as e:
            print(f"  ! falha ao listar {url}: {e}")
            return

        for nome, url_abs, eh_pasta in entradas:
            if eh_pasta:
                if depth >= ds["max_depth"]:
                    continue
                # no nível do ano, filtra pelos anos configurados
                if depth == 0 and ds["filtro_anos"]:
                    ano = re.match(r"^(\d{4})$", nome)
                    if ano and int(ano.group(1)) not in ds["filtro_anos"]:
                        continue
                    if not ano:
                        continue
                caminhar(url_abs, depth + 1)
            else:
                if regex.match(nome):
                    encontrados.append((nome, url_abs))

    caminhar(ds["listing_url"], 0)

    vistos, saida = set(), []
    for nome, url in encontrados:
        if nome not in vistos:
            vistos.add(nome)
            saida.append((nome, url))
    return sorted(saida)


def metadados_remotos(url: str) -> dict:
    """HEAD: decide se precisa baixar sem trazer o arquivo inteiro."""
    try:
        r = requests.head(url, headers=HEADERS, timeout=60, allow_redirects=True)
        r.raise_for_status()
        cl = r.headers.get("Content-Length")
        return {
            "content_length": int(cl) if cl and cl.isdigit() else None,
            "last_modified": r.headers.get("Last-Modified"),
            "etag": r.headers.get("ETag"),
        }
    except Exception as e:
        print(f"  ! HEAD falhou ({e}) - baixando por precaução")
        return {"content_length": None, "last_modified": None, "etag": None}

# COMMAND ----------

# =============================================================================
# DOWNLOAD -> VOLUME
# =============================================================================

def baixar_para_volume(url: str, nome_arquivo: str, subpasta: str) -> tuple:
    """Stream para /tmp, calcula sha256, copia para o Volume, limpa o tmp."""
    pasta_destino = f"{VOLUME_PATH}/{subpasta}"
    os.makedirs(pasta_destino, exist_ok=True)  # Volumes suportam subpastas normalmente

    tmp_path = f"/tmp/{nome_arquivo}"
    destino = f"{pasta_destino}/{nome_arquivo}"

    sha, total = hashlib.sha256(), 0
    with requests.get(url, headers=HEADERS, timeout=TIMEOUT, stream=True) as r:
        r.raise_for_status()
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    sha.update(chunk)
                    total += len(chunk)

    with open(tmp_path, "rb") as origem, open(destino, "wb") as dest:
        while True:
            buf = origem.read(1024 * 1024)
            if not buf:
                break
            dest.write(buf)

    os.remove(tmp_path)
    return total, sha.hexdigest()

# COMMAND ----------

# =============================================================================
# EXECUÇÃO
# =============================================================================

controle = carregar_controle()
novos_registros = []
resumo = {"baixados": [], "ignorados": [], "erros": []}

for ds in DATASETS:
    print(f"\n=== {ds['nome']} ===")
    arquivos = coletar_arquivos(ds)
    print(f"  {len(arquivos)} arquivo(s) encontrado(s) no portal")

    for nome, url in arquivos:
        meta = metadados_remotos(url)
        anterior = controle.get(nome)

        precisa = True
        if anterior:
            cl_ant, lm_ant, etag_ant = anterior
            igual_tam = meta["content_length"] is not None and meta["content_length"] == cl_ant
            igual_data = meta["last_modified"] is not None and meta["last_modified"] == lm_ant
            igual_etag = meta["etag"] is not None and meta["etag"] == etag_ant
            if (igual_tam and igual_data) or igual_etag:
                precisa = False

        if not precisa:
            resumo["ignorados"].append(nome)
            continue

        try:
            tamanho, sha256 = baixar_para_volume(url, nome, ds["subpasta"])
            print(f"  + {ds['subpasta']}/{nome}  ({tamanho/1024/1024:.2f} MB)")
            resumo["baixados"].append(nome)
            novos_registros.append((
                ds["nome"], nome, url, tamanho,
                meta["last_modified"], meta["etag"], sha256,
                datetime.now(timezone.utc),
            ))
        except Exception as e:
            print(f"  ! erro em {nome}: {e}")
            resumo["erros"].append((nome, str(e)))

registrar_controle(novos_registros)

print("\n=== RESUMO ===")
print(f"Baixados : {len(resumo['baixados'])} -> {resumo['baixados']}")
print(f"Ignorados: {len(resumo['ignorados'])} (sem alteração no servidor)")
print(f"Erros    : {len(resumo['erros'])} -> {resumo['erros']}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Conferência
# MAGIC O que está registrado no controle (último load de cada arquivo).

# COMMAND ----------

display(
    spark.table(CONTROL_TABLE)
         .orderBy(F.col("loaded_at").desc())
         .select("dataset", "file_name", "content_length", "last_modified", "loaded_at")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Como agendar (Databricks Workflows)
# MAGIC
# MAGIC 1. **Workflows** -> **Create job**
# MAGIC 2. Task: tipo **Notebook**, apontando para este notebook
# MAGIC 3. Compute: **Single node**, `Standard_DS3_v2` não se aplica na AWS —
# MAGIC    use `i3.xlarge` ou `m5d.large` (workspace em `us-east-2`).
# MAGIC    Access mode: **Single user** (obrigatório para escrever em Volumes do UC).
# MAGIC 4. **Schedule** -> Every day at 06:00 (America/Sao_Paulo)
# MAGIC 5. **Notifications** -> e-mail em caso de falha
# MAGIC
# MAGIC Permissões necessárias no Unity Catalog para o usuário/SP que roda o job:
# MAGIC ```sql
# MAGIC GRANT READ VOLUME, WRITE VOLUME ON VOLUME voebem.bronze.arquivos TO `<principal>`;
# MAGIC GRANT USE CATALOG ON CATALOG voebem TO `<principal>`;
# MAGIC GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT ON SCHEMA voebem.bronze TO `<principal>`;
# MAGIC ```

# COMMAND ----------

dbutils.notebook.exit(json.dumps({
    "baixados": resumo["baixados"],
    "ignorados": len(resumo["ignorados"]),
    "erros": [e[0] for e in resumo["erros"]],
}))
